# UrduStack — Train Risk Scorer (LoRA XLM-RoBERTa)

Run this notebook in Google Colab free-tier GPU (T4).

**What it does**
1. **Builds a frequency map** from Roman-Urdu-Parl (6.37M parallel sentences) to expand normalization dictionary from ~467 → 50k+ words
2. **Downloads 4 datasets** from Hugging Face automatically:
   - **Roman-Urdu-Toxic-Corpus** (72.7k rows, CC-BY-4.0) — toxicity labels
   - **Roman-Urdu Hate Speech** (7k rows, MIT) — hate speech labels
   - **Urdu Spam Dataset** (3k rows, MIT) — spam labels
   - **Synthetic Scam Data** (1,500 generated examples) — scam-specific patterns
3. **Merges** them into a combined training set with class balance audit
4. **Fine-tunes** `xlm-roberta-base` on the full combined dataset using LoRA (5 epochs)
5. **Evaluates** Whisper on Urdu speech benchmark (WER baseline)
6. Saves a small LoRA adapter to `models/risk_lora/`
7. Computes a temperature-scaling value and writes it to `models/temperature.txt`

**Expected total runtime on free Colab T4**
- Setup + installs: ~3–5 min
- Frequency map build: ~5–10 min
- Dataset downloads + merge: ~3–5 min
- Training (5 epochs, ~85k samples, batch 16): ~60–90 min
- Temperature calibration + test eval: ~2–3 min
- Whisper eval (optional, streaming): ~5–10 min
- **Total: ~80–120 minutes**

**You do not need to download any dataset manually.** All cells fetch data directly from Hugging Face.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies (pinned versions to avoid PEFT / Python 3.13 crashes).
# Colab already has torch installed; we only add what the training script needs.
#
# IMPORTANT: drop torchao first. Colab ships torchao 0.10, but PEFT >=0.14
# aborts on import if it finds anything below 0.16. We don't need torchao
# for LoRA fine-tuning, so uninstalling it is the cleanest fix.
#
# NOTE: pandas is left unpinned. Colab's pyarrow needs pandas 3.x.
# pip will warn about google-colab/cudf wanting pandas<2.4 — those are
# harmless warnings and do NOT affect our training pipeline.
!pip uninstall -y torchao 2>/dev/null
!pip install -q -U \
  "transformers>=4.46.0" \
  "datasets>=3.1.0" \
  "peft>=0.13.2" \
  "accelerate>=1.1.0" \
  "openai-whisper>=20231117" \
  scikit-learn pandas

# Sanity-check the versions that matter
import peft, transformers, datasets
print(f"peft={peft.__version__}  transformers={transformers.__version__}  datasets={datasets.__version__}")

In [ ]:
# Optional: mount Google Drive so the trained adapter is saved permanently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the UrduStack repo (idempotent — handles re-runs and nested clones).
import os, shutil

# If a previous run accidentally cloned UrduStack inside UrduStack, remove it.
if os.path.exists('UrduStack/UrduStack'):
    shutil.rmtree('UrduStack/UrduStack')
    print('Removed nested UrduStack/UrduStack')

if os.path.exists('UrduStack'):
    %cd UrduStack
    !git pull
else:
    !git clone https://github.com/munazat/UrduStack.git
    %cd UrduStack

!pwd   # should print /content/UrduStack

In [ ]:
# PRIORITY 1: Build Roman-Urdu frequency map from Roman-Urdu-Parl (6.37M parallel sentences).
# This expands the normalization dictionary from ~467 words to 50k+.
# Source: https://huggingface.co/datasets/Mavkif/Roman-Urdu-Parl-split
# Columns: "Urdu text", "Roman-Urdu text" | License: CC-BY-4.0
# Runs on CPU — no GPU needed. Takes ~5-10 minutes on Colab.
import os
from datasets import load_dataset

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

parl_csv = 'data/raw/roman_urdu_parl.csv'

if not os.path.exists(parl_csv):
    print('Downloading Roman-Urdu-Parl from Hugging Face (6.37M rows, ~200MB)...')
    ds = load_dataset('Mavkif/Roman-Urdu-Parl-split', split='train')
    df_parl = ds.to_pandas()
    df_parl.to_csv(parl_csv, index=False)
    print(f'Downloaded {len(df_parl)} rows. Columns: {list(df_parl.columns)}')
    print(f'Saved to {parl_csv}')
    del df_parl  # free memory
else:
    print(f'{parl_csv} already present — skipping download.')

# Build frequency map using the updated script (handles actual column names).
print('\nBuilding frequency map (this takes ~5-10 minutes)...')
!python scripts/build_normalizer_map.py \
  --input data/raw/roman_urdu_parl.csv \
  --output data/processed/roman_urdu_freq.json \
  --min_count 2 \
  --chunk_size 500000

import json
if os.path.exists('data/processed/roman_urdu_freq.json'):
    with open('data/processed/roman_urdu_freq.json', 'r', encoding='utf-8') as f:
        freq_map = json.load(f)
    print(f'\nFrequency map built: {len(freq_map)} Roman-Urdu -> Urdu mappings')
    sample = list(freq_map.items())[:10]
    print('Sample mappings:')
    for r, u in sample:
        print(f'  {r} -> {u}')
else:
    print('ERROR: frequency map file not created!')

In [ ]:
# Download the Roman-Urdu-Toxic-Corpus from Hugging Face and save as CSV.
# Source: https://huggingface.co/datasets/hafiz-hassaan-saeed/Roman-Urdu-Toxic-Corpus
# 72,771 rows | columns: Roman_Urdu, Toxic | license: CC-BY-4.0
import os
from datasets import load_dataset

os.makedirs('data/raw', exist_ok=True)

if not os.path.exists('data/raw/PURUTT.csv'):
    print('Downloading Roman-Urdu-Toxic-Corpus from Hugging Face...')
    ds = load_dataset('hafiz-hassaan-saeed/Roman-Urdu-Toxic-Corpus', split='train')
    df = ds.to_pandas()

    # Rename columns to match what the training script expects
    df = df.rename(columns={'Roman_Urdu': 'text', 'Toxic': 'label'})

    df.to_csv('data/raw/PURUTT.csv', index=False)
    print(f'Downloaded {len(df)} rows. Saved to data/raw/PURUTT.csv')
else:
    print('PURUTT.csv already present — skipping download.')

In [ ]:
# Download additional datasets and merge into a combined training CSV.
# 1. Roman-Urdu Hate Speech (7k rows, MIT) — broader toxicity coverage
# 2. Urdu Spam Dataset (3k rows, MIT) — spam patterns
# 3. Synthetic Scam Data (1500 generated examples) — scam-specific patterns
#
# IMPORTANT: Label convention across all datasets: 1=toxic/spam, 0=clean.
# Each dataset is verified and normalized to this convention before merging.
import os
import pandas as pd
from datasets import load_dataset

frames = []

# --- PURUTT (already downloaded in the previous cell) ---
purutt = pd.read_csv('data/raw/PURUTT.csv')
purutt = purutt[['text', 'label']].dropna()
purutt['label'] = purutt['label'].astype(int)
print(f"PURUTT: {len(purutt)} rows | toxic: {purutt['label'].sum()} | clean: {(purutt['label']==0).sum()}")
frames.append(purutt)

# --- Roman-Urdu Hate Speech (Coarse_Grained subset) ---
# VERIFIED label convention in dataset: 0=Abusive/Offensive, 1=Normal
# We INVERT to match our convention: 1=toxic, 0=clean
try:
    hate_ds = load_dataset('community-datasets/roman_urdu_hate_speech', 'Coarse_Grained', split='train')
    hate_df = hate_ds.to_pandas()
    hate_df = hate_df.rename(columns={'tweet': 'text'})
    # INVERT: dataset 0=abusive→our 1=toxic; dataset 1=normal→our 0=clean
    hate_df['label'] = 1 - hate_df['label'].astype(int)
    hate_df = hate_df[['text', 'label']].dropna()
    print(f"Hate Speech: {len(hate_df)} rows | toxic: {hate_df['label'].sum()} | clean: {(hate_df['label']==0).sum()}")
    frames.append(hate_df)
except Exception as e:
    print(f"WARNING: Could not load hate speech dataset: {e}")
    print("Continuing with other datasets.")

# --- Urdu Spam Dataset ---
# VERIFIED label convention: 1=Spam, 0=Not Spam (already matches our convention)
try:
    spam_ds = load_dataset('hamza-amin/urdu-spam-dataset', split='train')
    spam_df = spam_ds.to_pandas()
    spam_df = spam_df[['text', 'label']].dropna()
    spam_df['label'] = spam_df['label'].astype(int)
    print(f"Spam: {len(spam_df)} rows | spam: {spam_df['label'].sum()} | clean: {(spam_df['label']==0).sum()}")
    frames.append(spam_df)
except Exception as e:
    print(f"WARNING: Could not load spam dataset: {e}")
    print("Continuing with available datasets.")

# --- PRIORITY 2: Synthetic Scam Data ---
# Generate 1500 scam examples covering job scams, phishing, investment fraud, etc.
# These are template-based with randomized amounts/numbers for variety.
print("\nGenerating synthetic scam data...")
!python scripts/generate_scam_data.py \
  --output data/raw/synthetic_scam.csv \
  --n_samples 1500 \
  --seed 42

if os.path.exists('data/raw/synthetic_scam.csv'):
    scam_gen = pd.read_csv('data/raw/synthetic_scam.csv')
    scam_gen = scam_gen[['text', 'label']].dropna()
    scam_gen['label'] = scam_gen['label'].astype(int)
    print(f"Synthetic Scam: {len(scam_gen)} rows | all scam: {scam_gen['label'].sum()}")
    frames.append(scam_gen)
else:
    print("WARNING: synthetic scam data not generated!")

# --- Merge and save ---
combined = pd.concat(frames, ignore_index=True)
combined = combined.drop_duplicates(subset=['text']).sample(frac=1, random_state=42).reset_index(drop=True)
combined.to_csv('data/raw/combined_risk.csv', index=False)

total = len(combined)
toxic = combined['label'].sum()
clean = (combined['label'] == 0).sum()
ratio = toxic / max(clean, 1)
print(f"\n{'='*50}")
print(f"Combined dataset: {total} rows")
print(f"  Toxic/Spam (label=1): {toxic} ({toxic/total*100:.1f}%)")
print(f"  Clean      (label=0): {clean} ({clean/total*100:.1f}%)")
print(f"  Toxic/Clean ratio:    {ratio:.2f}")
if ratio < 0.3 or ratio > 3.0:
    print("  WARNING: significant class imbalance — training script will use class weighting.")
print(f"{'='*50}")
print("Saved to data/raw/combined_risk.csv")

In [ ]:
# PRIORITY 3: Class balance audit — visualize label distribution and source breakdown.
# This cell does NOT modify data — it only reports. If imbalance is severe,
# the training script's class-weighted loss will compensate automatically.
import pandas as pd
import matplotlib.pyplot as plt

combined = pd.read_csv('data/raw/combined_risk.csv')

print(f"Total samples: {len(combined):,}")
print(f"Label distribution:")
label_counts = combined['label'].value_counts().sort_index()
for label, count in label_counts.items():
    pct = count / len(combined) * 100
    tag = "toxic/spam" if label == 1 else "clean"
    print(f"  label={label} ({tag}): {count:,} ({pct:.1f}%)")

# Text length statistics
combined['text_len'] = combined['text'].str.len()
print(f"\nText length stats:")
print(f"  Mean:   {combined['text_len'].mean():.0f} chars")
print(f"  Median: {combined['text_len'].median():.0f} chars")
print(f"  Min:    {combined['text_len'].min()} chars")
print(f"  Max:    {combined['text_len'].max()} chars")
print(f"  Std:    {combined['text_len'].std():.0f} chars")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(
    label_counts.values,
    labels=[f"Clean (label=0)\n{label_counts[0]:,}", f"Toxic/Spam (label=1)\n{label_counts[1]:,}"],
    autopct='%1.1f%%',
    colors=['#4CAF50', '#F44336'],
    startangle=90,
)
axes[0].set_title('Label Distribution')

# Text length histogram by class
for label, color, tag in [(0, '#4CAF50', 'Clean'), (1, '#F44336', 'Toxic/Spam')]:
    subset = combined[combined['label'] == label]['text_len']
    axes[1].hist(subset, bins=50, alpha=0.6, label=f'{tag} (n={len(subset):,})', color=color)
axes[1].set_xlabel('Text Length (chars)')
axes[1].set_ylabel('Count')
axes[1].set_title('Text Length by Class')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/processed/class_balance_audit.png', dpi=100, bbox_inches='tight')
plt.show()

# Imbalance check
toxic_ratio = label_counts[1] / len(combined)
if toxic_ratio < 0.1:
    print("\n⚠️ SEVERE IMBALANCE: toxic class is <10% — training may struggle with recall.")
    print("   The class-weighted loss in train_risk_model.py will compensate.")
elif toxic_ratio > 0.9:
    print("\n⚠️ SEVERE IMBALANCE: clean class is <10% — model may over-predict toxicity.")
else:
    print(f"\n✓ Class balance acceptable: toxic ratio = {toxic_ratio:.1%}")

In [ ]:
# Train on combined dataset (5 epochs)
# Dataset is ~83k rows, so we split: 70k train / 5k val / 5k test
!python scripts/train_risk_model.py \
  --data_path data/raw/combined_risk.csv \
  --output_dir models/risk_lora \
  --max_samples 70000 \
  --val_samples 5000 \
  --test_samples 5000 \
  --num_epochs 5 \
  --batch_size 16

In [ ]:
# Check outputs
import os
print('Adapter files:', os.listdir('models/risk_lora'))
if os.path.exists('models/temperature.txt'):
    print('Temperature:', open('models/temperature.txt').read().strip())

In [ ]:
# PRIORITY 4: Evaluate Whisper on Urdu speech benchmark (WER metric).
# This is EVALUATION ONLY — no ASR training. Tests how well pretrained
# Whisper performs on Urdu speech as a baseline for the STT pipeline.
#
# Primary: humairawan/UrduSpeech (has audio + text columns, train/test splits)
# Fallback: Common Voice Urdu (requires HF token for gated access)
import subprocess
subprocess.run(['pip', 'install', '-q', 'jiwer', 'soundfile'], capture_output=True)

import torch
import whisper
from datasets import load_dataset
from jiwer import wer, cer
import numpy as np

print("Loading Whisper base model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.load_model("base", device=device)
print(f"Whisper loaded on {device}")

hypotheses = []
references = []
n_eval = 50  # evaluate on 50 samples to keep runtime reasonable

def evaluate_samples(eval_iter, n, source_name):
    """Run Whisper on an iterable of {audio, text} samples."""
    hyps, refs = [], []
    for i, sample in enumerate(eval_iter):
        if i >= n:
            break
        audio = sample["audio"]
        if isinstance(audio, dict):
            audio_array = np.array(audio["array"], dtype=np.float32)
        else:
            audio_array = np.array(audio, dtype=np.float32)

        if len(audio_array.shape) > 1:
            audio_array = audio_array.mean(axis=1)

        result = whisper_model.transcribe(audio_array, language="ur", fp16=(device == "cuda"))
        hyp = result.get("text", "").strip()
        ref = sample.get("text", "").strip()
        if ref:
            hyps.append(hyp)
            refs.append(ref)
        if (i + 1) % 10 == 0:
            print(f"  [{source_name}] Processed {i+1}/{n} samples...")
    return hyps, refs

# --- Try primary dataset: humairawan/UrduSpeech (streaming to avoid 28GB download) ---
try:
    print("Trying humairawan/UrduSpeech (streaming test split)...")
    ds = load_dataset("humairawan/UrduSpeech", split="test", streaming=True)
    hypotheses, references = evaluate_samples(iter(ds), n_eval, "UrduSpeech")
except Exception as e:
    print(f"humairawan/UrduSpeech failed: {e}")

# --- Fallback: Common Voice Urdu ---
if not references:
    try:
        print("\nTrying Common Voice Urdu (streaming test split)...")
        ds = load_dataset("mozilla-foundation/common_voice_17_0", "ur",
                          split="test", streaming=True, trust_remote_code=True)
        def remap_common_voice(it):
            for sample in it:
                yield {"audio": sample["audio"], "text": sample["sentence"]}
        hypotheses, references = evaluate_samples(remap_common_voice(iter(ds)), n_eval, "CommonVoice")
    except Exception as e:
        print(f"Common Voice Urdu failed: {e}")

# --- Report results ---
if references:
    word_error_rate = wer(references, hypotheses)
    char_error_rate = cer(references, hypotheses)
    print(f"\n{'='*50}")
    print(f"Whisper Urdu Speech Evaluation ({len(references)} samples):")
    print(f"  Word Error Rate (WER): {word_error_rate:.2%}")
    print(f"  Char Error Rate (CER): {char_error_rate:.2%}")
    print(f"{'='*50}")

    print("\nSample predictions:")
    for ref, hyp in list(zip(references, hypotheses))[:5]:
        print(f"  REF: {ref}")
        print(f"  HYP: {hyp}")
        print()
else:
    print("\nCould not load any Urdu speech evaluation dataset.")
    print("This is non-critical — the STT pipeline still works with Whisper for inference.")
    print("To evaluate manually: record Urdu audio and compare Whisper output to reference text.")

In [ ]:
# Optional: push adapter to Hugging Face Hub
# !huggingface-cli login
# !python scripts/train_risk_model.py \
#   --data_path data/raw/PURUTT.csv \
#   --output_dir models/risk_lora \
#   --push_to_hub \
#   --hub_model_id your-username/urdustack-risk-lora

In [ ]:
# Optional: copy results to Drive (only if Drive was mounted)
import os, shutil

drive_dest = '/content/drive/MyDrive/urdustack_models'
if os.path.exists('/content/drive/MyDrive'):
    shutil.copytree('models', drive_dest, dirs_exist_ok=True)
    print('Copied models to', drive_dest)
else:
    print('Drive not mounted — skipping copy. Use the Files panel to download models/ manually.')

In [ ]:
# Launch live demo with a public URL (share=True gives you a *.gradio.live link)
# The model loads on Colab's T4 GPU — inference is fast.
# This URL works as long as the Colab notebook is running (~72 hours).
import sys
sys.path.insert(0, '/content/UrduStack')

import gradio as gr
from playground import analyze, transcribe_and_analyze, find_entities, simplify_text

with gr.Blocks(title="UrduStack — Code-Switch-Aware Urdu NLP", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# UrduStack\n"
        "Code-switch-aware Urdu NLP: normalization, risk scoring, "
        "speech-to-text, NER, and lexical simplification in one pipeline."
    )

    with gr.Tabs():
        with gr.Tab("Text Analysis"):
            text_input = gr.Textbox(
                label="Input (Urdu / Roman Urdu / English mix)",
                placeholder="yar bhai I'm bohat pareshan aaj...",
                lines=3,
            )
            text_btn = gr.Button("Analyze", variant="primary")
            with gr.Row():
                norm_out = gr.Textbox(label="Normalized Urdu", interactive=False)
                score_out = gr.Textbox(label="Risk Score", interactive=False)
            analysis_out = gr.Markdown(label="Analysis")
            text_btn.click(analyze, inputs=[text_input], outputs=[norm_out, score_out, analysis_out])

        with gr.Tab("Speech-to-Text"):
            audio_input = gr.Audio(label="Record or upload Urdu audio", type="filepath")
            audio_btn = gr.Button("Transcribe & Analyze", variant="primary")
            transcript_out = gr.Textbox(label="Transcription", interactive=False)
            audio_norm_out = gr.Textbox(label="Normalized Urdu", interactive=False)
            audio_analysis_out = gr.Markdown(label="Analysis")
            audio_conf_out = gr.Textbox(label="Speech Confidence", interactive=False)
            audio_btn.click(
                transcribe_and_analyze,
                inputs=[audio_input],
                outputs=[transcript_out, audio_norm_out, audio_analysis_out, audio_conf_out],
            )

        with gr.Tab("Named Entities"):
            ner_input = gr.Textbox(
                label="Input text",
                placeholder="Imran Khan ne Lahore mein PTI ki rally ki...",
                lines=3,
            )
            ner_btn = gr.Button("Find Entities", variant="primary")
            ner_out = gr.Markdown(label="Entities")
            ner_btn.click(find_entities, inputs=[ner_input], outputs=[ner_out])

        with gr.Tab("Simplify"):
            simp_input = gr.Textbox(
                label="Urdu text to simplify",
                placeholder="حکومت نے ضروری تعلیم کے لیے نئی معلومات جاری کی...",
                lines=3,
            )
            simp_btn = gr.Button("Simplify", variant="primary")
            simp_out = gr.Textbox(label="Simplified text", interactive=False)
            simp_changes = gr.Markdown(label="Changes")
            simp_btn.click(simplify_text, inputs=[simp_input], outputs=[simp_out, simp_changes])

    gr.Examples(
        examples=[
            ["yar mujhe pareshan mat karo bro"],
            ["job available, 50000 per week, send processing fee"],
            ["aaj weather bohat achha hai"],
            ["bhai ye to scam lag raha hai, paise mat bhejo"],
            ["free iphone jeetny k liye link click karein"],
        ],
        inputs=[text_input],
    )

demo.launch(share=True)